In [2]:
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive")

PROJECT_PATH = Path("/content/drive/MyDrive/Flower_Project")
DATASETS_DIR = PROJECT_PATH / "datasets"
RUNS_DIR = PROJECT_PATH / "training_results"
MODELS_DIR = PROJECT_PATH / "models"
VAL_DIR = PROJECT_PATH / "validation_results"
PREDICT_DIR = PROJECT_PATH / "prediction_results"

for p in [PROJECT_PATH, DATASETS_DIR, RUNS_DIR, MODELS_DIR, VAL_DIR, PREDICT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("프로젝트 경로:", PROJECT_PATH)

Mounted at /content/drive
프로젝트 경로: /content/drive/MyDrive/Flower_Project


In [3]:
!pip install -q ultralytics roboflow pyyaml

import torch

print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 63.8 MB/s eta 0:00:00
CUDA 사용 가능: True
GPU: Tesla T4
Thu May 21 10:52:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       

In [4]:
import os
import shutil
from pathlib import Path
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

if not ROBOFLOW_API_KEY:
    raise ValueError("Colab Secrets에 ROBOFLOW_API_KEY를 먼저 등록하세요")

WORKSPACE = "minseo-kim-nw4w2"
PROJECT = "bouquet-total"
VERSION = 3

DATASET_DRIVE_DIR = DATASETS_DIR / f"{PROJECT}-v{VERSION}-yolov11"

FORCE_REDOWNLOAD = False

if FORCE_REDOWNLOAD and DATASET_DRIVE_DIR.exists():
    shutil.rmtree(DATASET_DRIVE_DIR)

if not (DATASET_DRIVE_DIR / "data.yaml").exists():
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)

    dataset = version.download(
        "yolov11",
        location=str(DATASET_DRIVE_DIR)
    )

    print("다운로드 완료:", dataset.location)
else:
    print("이미 다운로드된 데이터셋 사용:", DATASET_DRIVE_DIR)

loading Roboflow workspace...
loading Roboflow project...


Extracting Dataset Version Zip to /content/drive/MyDrive/Flower_Project/datasets/bouquet-total-v3-yolov11 in yolov11:: 100%|██████████| 62303/62303 [15:29<00:00, 67.03it/s]


다운로드 완료: /content/drive/MyDrive/Flower_Project/datasets/bouquet-total-v3-yolov11


In [5]:
import shutil
from pathlib import Path

LOCAL_DATASET_DIR = Path("/content/flower_data_v3")

if LOCAL_DATASET_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_DIR)

shutil.copytree(DATASET_DRIVE_DIR, LOCAL_DATASET_DIR)

print("로컬 복사 완료:", LOCAL_DATASET_DIR)
print("파일 목록:", os.listdir(LOCAL_DATASET_DIR))

로컬 복사 완료: /content/flower_data_v3
파일 목록: ['valid', 'README.roboflow.txt', 'test', 'train', 'data.yaml', 'README.dataset.txt']


In [6]:
import yaml
from pathlib import Path

original_yaml_path = LOCAL_DATASET_DIR / "data.yaml"
fixed_yaml_path = LOCAL_DATASET_DIR / "data_fixed.yaml"

with open(original_yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data.get("names")
nc = data.get("nc", len(names) if names else 0)

fixed_data = {
    "path": str(LOCAL_DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": nc,
    "names": names,
}

with open(fixed_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_data, f, allow_unicode=True, sort_keys=False)

print("수정된 YAML:", fixed_yaml_path)
print(yaml.safe_dump(fixed_data, allow_unicode=True, sort_keys=False))

수정된 YAML: /content/flower_data_v3/data_fixed.yaml
path: /content/flower_data_v3
train: train/images
val: valid/images
test: test/images
nc: 88
names:
- alstroemeria
- amaryllis
- anthurium
- azalea
- bee_balm
- bellflower
- blackberry_lily
- blanket_flower
- bougainvillea
- bromeliad
- calla_lily
- camellia
- canna_lily
- canterbury_bells
- cape_flower
- carnation
- cattleya
- celosia
- chamomile
- chrysanthemum
- clematis
- columbine
- coneflower
- cosmos
- cyclamen
- daffodil
- dahlia
- daisy
- desert_rose
- doraji
- eryngo
- feather_celosia
- foxglove
- freesia
- fritillaria
- garden_phlox
- gaura
- gazania
- gentian
- geranium
- gladiolus
- globe_thistle
- gloriosa_lily
- gyeongyeopduran
- gypsophila
- hellebores
- hibiscus
- hyacinth
- hydrangea
- iris
- ixora
- japanese_anemone
- kalanchoe
- lily
- lisianthus
- magnolia
- marigold
- masterwort
- mexican_petunia
- monkshood
- morning_glory
- mulmangcho
- nasturtium
- nigella
- orchid
- osteospermum
- pansy
- passion_flower
- peony

In [7]:
from pathlib import Path

def count_files(path, exts):
    path = Path(path)
    if not path.exists():
        return 0
    return len([p for p in path.iterdir() if p.suffix.lower() in exts])

img_exts = {".jpg", ".jpeg", ".png", ".webp"}
label_exts = {".txt"}

for split in ["train", "valid", "test"]:
    img_count = count_files(LOCAL_DATASET_DIR / split / "images", img_exts)
    label_count = count_files(LOCAL_DATASET_DIR / split / "labels", label_exts)
    print(f"{split}: images={img_count}, labels={label_count}")

print("클래스 수:", nc)
print("클래스 목록:", names)

train: images=27688, labels=27688
valid: images=1731, labels=1731
test: images=1730, labels=1730
클래스 수: 88
클래스 목록: ['alstroemeria', 'amaryllis', 'anthurium', 'azalea', 'bee_balm', 'bellflower', 'blackberry_lily', 'blanket_flower', 'bougainvillea', 'bromeliad', 'calla_lily', 'camellia', 'canna_lily', 'canterbury_bells', 'cape_flower', 'carnation', 'cattleya', 'celosia', 'chamomile', 'chrysanthemum', 'clematis', 'columbine', 'coneflower', 'cosmos', 'cyclamen', 'daffodil', 'dahlia', 'daisy', 'desert_rose', 'doraji', 'eryngo', 'feather_celosia', 'foxglove', 'freesia', 'fritillaria', 'garden_phlox', 'gaura', 'gazania', 'gentian', 'geranium', 'gladiolus', 'globe_thistle', 'gloriosa_lily', 'gyeongyeopduran', 'gypsophila', 'hellebores', 'hibiscus', 'hyacinth', 'hydrangea', 'iris', 'ixora', 'japanese_anemone', 'kalanchoe', 'lily', 'lisianthus', 'magnolia', 'marigold', 'masterwort', 'mexican_petunia', 'monkshood', 'morning_glory', 'mulmangcho', 'nasturtium', 'nigella', 'orchid', 'osteospermum', 

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = str(fixed_yaml_path)

RUN_NAME = "bouquet_yolo11s_v3_768"

model = YOLO("yolo11s.pt")

results = model.train(
    data=DATA_YAML,

    # train 27,688장이고 Roboflow에서 이미 2x augmentation 했으므로 120은 너무 김
    epochs=60,
    patience=15,

    # Roboflow resize가 960이지만 Colab 시간 고려해서 768 유지
    imgsz=704,
    batch=16,
    workers=4,
    device=0,

    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,

    pretrained=True,
    optimizer="auto",
    cos_lr=True,

    # Roboflow에서 색상 augmentation 이미 적용했으므로 YOLO 색상 augmentation OFF
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    # Roboflow에서 crop/rotation 이미 적용했으므로 YOLO 기하 augmentation OFF
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    # Roboflow에서 horizontal flip 이미 적용했으므로 OFF
    fliplr=0.0,
    flipud=0.0,

    # Roboflow augmentation과 중복 방지
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    cache=False,
    amp=True,
    plots=True,
    save_period=10,

    seed=42,
    deterministic=True,
)

Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/flower_data_v3/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=704, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=bouquet_yolo11s_v3_768-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_MODEL_PATH = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

model = YOLO(str(BEST_MODEL_PATH))

metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=960,
    iou=0.65,
    plots=True,
    project=str(VAL_DIR),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)

print("검증 완료")
print("best.pt:", BEST_MODEL_PATH)

In [ ]:
import shutil
from pathlib import Path

FINAL_MODEL_PATH = MODELS_DIR / "bouquet_yolo11s_v3_best.pt"

shutil.copy2(BEST_MODEL_PATH, FINAL_MODEL_PATH)

print("최종 모델 저장 완료:")
print(FINAL_MODEL_PATH)

테스트

In [ ]:
from ultralytics import YOLO

TEST_IMAGE_DIR = PROJECT_PATH / "test_images"

model = YOLO(str(FINAL_MODEL_PATH))

results = model.predict(
    source=str(TEST_IMAGE_DIR),
    imgsz=960,
    conf=0.25,
    iou=0.65,
    agnostic_nms=False,
    max_det=150,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(PREDICT_DIR),
    name="bouquet_yolo11m_v3_predictions",
    exist_ok=True,
)

print("예측 결과 저장 위치:")
print(PREDICT_DIR / "bouquet_yolo11m_v3_predictions")